In [18]:
import pandas as pd
from datasets import load_dataset
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

### Mapping each pair of context - query into a dict for evaluation metrics ###

In [19]:
class MapTruth:
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def project(self):
        documents = self.df['context'].unique().tolist()
        doc_to_idx = {text: i for i, text in enumerate(documents)}

        queries = self.df['query'].unique().tolist()
        query_to_idx = {text: i for i, text in enumerate(queries)}

        ground_truth = {}
        
        grouped = self.df.groupby('query')['context'].apply(list).to_dict()

        for query_text, context_list in grouped.items():
            q_idx = query_to_idx[query_text]
            # Convert the text contexts into their corresponding document indices
            d_indices = [doc_to_idx[ctx] for ctx in context_list]
            ground_truth[q_idx] = d_indices

        return queries, documents, ground_truth

### Evaluation metrics: ###
* HitRate @k
* Recall @k

In [20]:
def evaluate(models, queries, documents, ground_truth, k_values=[10, 20]):
    results = {}
    for model in models:
        doc_embeddings   = np.array(model.encode(documents))
        query_embeddings = np.array(model.encode(queries))
        sim_matrix = cosine_similarity(query_embeddings, doc_embeddings)

        metrics = {"similarity_matrix": sim_matrix}
        for k in k_values:
            metrics[f"Recall@{k}"]  = round(_recall(sim_matrix, ground_truth, k), 4)
            metrics[f"HitRate@{k}"] = round(_hit_rate(sim_matrix, ground_truth, k), 4)

        results[model.name] = metrics
    return results


def _recall(sim_matrix, ground_truth, k):
    scores = [
        sum(1 for i in np.argsort(sim_matrix[q])[::-1][:k] if i in correct) / len(correct)
        for q, correct in ground_truth.items()
    ]
    return np.mean(scores) if scores else 0


def _hit_rate(sim_matrix, ground_truth, k):
    hits = [
        any(i in np.argsort(sim_matrix[q])[::-1][:k] for i in correct)
        for q, correct in ground_truth.items()
    ]
    return np.mean(hits) if hits else 0

### QASPER Dataset Loader

In [21]:
import pandas as pd
from datasets import load_dataset

def load_qasper_test() -> pd.DataFrame:
    parquet_url = "https://huggingface.co/datasets/allenai/qasper/resolve/refs%2Fconvert%2Fparquet/qasper/test/0000.parquet"
    
    ds = load_dataset("parquet", data_files={"test": parquet_url}, split="test")
    
    rows = []
    for paper in ds:
        qas_questions = paper["qas"]["question"]
        qas_answers = paper["qas"]["answers"]
        
        for question, answer_set in zip(qas_questions, qas_answers):
            for answer_dict in answer_set["answer"]:
                if answer_dict["unanswerable"]:
                    continue
                
                for evidence in answer_dict["evidence"]:
                    text = evidence.strip()
                    if text and not text.startswith("FLOAT SELECTED"):
                        rows.append({"query": question, "context": text})
                        
    df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["query", "context"])
        .reset_index(drop=True)
    )
    
    return df

In [22]:
df = load_qasper_test()
print(df.head(2))

                               query  \
0  How big is the ANTISCAM dataset?    
1           How is intent annotated?   

                                             context  
0  To enrich available non-collaborative task dat...  
1  To enrich publicly available non-collaborative...  


In [23]:
mapping = MapTruth(df)
queries, docs, ground_truth = mapping.project()

In [24]:
for d, q in list(ground_truth.items())[:1]:
    print(f"Query: \n{queries[d]}")
    print(f"Context:")
    for i in q:
        print(docs[i])
    print()

Query: 
Are LSA-reduced n-gram features considered hand-crafted features?
Context:
In this work, a neural network-based model namely RNN with attention (RNNwA) is proposed on the task of gender prediction from tweets. The proposed model is further improved by hand-crafted features which are obtained by LSA-reduced n-grams and concatenated with the neural representation from RNNwA. User representations that is the result of this model is then fed to a fully-connected layer to make prediction. This improved model achieved state-of-the-art accuracy on English and has a competitive performance on Spanish and Arabic.
The traditional approach to gender prediction problem is extracting a useful set of hand-crafted features and then feeding them into a standard classification algorithm. In their study, BIBREF0 work with the style-based features of message length, stop word usage, frequency of smiley etc. and use different classifiers such as k-nearest neighbor, naive bayes, covering rules, and

In [25]:
from embeddings.google import GeminiEmbedding
from embeddings.fastEmbed import FastEmbedding

In [26]:
fastembed_models = ["BAAI/bge-large-en-v1.5", "BAAI/bge-base-en",
                    "snowflake/snowflake-arctic-embed-m",
                    "snowflake/snowflake-arctic-embed-l","jinaai/jina-embeddings-v2-base-en",
                    "intfloat/multilingual-e5-large",
                    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"]
google = ["text-embedding-2"]
embed_models = {
    "fastembed": fastembed_models,
    "google":google
}

In [ ]:
import torch
import gc

results = {}

for name in embed_models["fastembed"]:
    try:
        model = FastEmbedding(name)
        model_result = evaluate([model], queries, docs, ground_truth, k_values=[1, 5])
        results.update(model_result)
    except Exception as e:
        print(f"Error evaluating {name}: {e}")
    finally:
        if 'model' in locals():
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for model_name, metrics in results.items():
    print(f"Model: {model_name}")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value}")